## Reading CSV Files from S3

This notebook demonstrates two approaches to read CSV files:

1. **PySpark DataFrame API** - Using `spark.read.csv()`
2. **SQL read_files() Function** - Using SQL syntax with `read_files()`

**S3 Path:** `/Volumes/dev_finance/staging/data_export/csv/`

We'll explore various properties and options for each approach.

## CSV Reading Options - Complete Reference

### Common Options for Both PySpark and SQL

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| **header** | boolean | false | **Use first row as column names**<br>• `true`: First row contains column names<br>• `false`: Columns named as `_c0`, `_c1`, etc. |
| **inferSchema** | boolean | false | **Automatically detect column data types**<br>• `true`: Scans data to determine types (int, string, timestamp, etc.)<br>• `false`: All columns read as strings<br>• **Note**: Requires extra data scan, slower for large files |
| **delimiter** | string | `,` | **Character separating fields**<br>• Common: `,` (comma), `\t` (tab), `|` (pipe)<br>• Also called "separator" |
| **quote** | string | `"` | **Character used to quote fields**<br>• Fields containing delimiters must be quoted<br>• Example: `"Smith, John",25` |
| **escape** | string | `\` | **Character to escape special characters**<br>• Used before quote character: `\"` |
| **nullValue** | string | empty | **String representing NULL values**<br>• Common: `NULL`, `NA`, `N/A`, empty string<br>• Example: If file has "NULL", set this to "NULL" |
| **mode** | string | PERMISSIVE | **How to handle malformed records**<br>• `PERMISSIVE`: Sets malformed fields to null, puts record in `_corrupt_record`<br>• `DROPMALFORMED`: Drops rows with malformed data<br>• `FAILFAST`: Throws exception on malformed data |

### Additional PySpark-Specific Options

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| **encoding** | string | UTF-8 | Character encoding (UTF-8, ISO-8859-1, etc.) |
| **dateFormat** | string | yyyy-MM-dd | Date format pattern |
| **timestampFormat** | string | yyyy-MM-dd'T'HH:mm:ss[.SSS][XXX] | Timestamp format pattern |
| **multiLine** | boolean | false | Parse CSV spanning multiple lines |
| **ignoreLeadingWhiteSpace** | boolean | false | Skip leading whitespace from values |
| **ignoreTrailingWhiteSpace** | boolean | false | Skip trailing whitespace from values |
| **comment** | string | none | Character for comment lines (skips those lines) |
| **maxColumns** | int | 20480 | Maximum number of columns |
| **pathGlobFilter** | string | none | Glob pattern to filter files (e.g., `*.csv`) |

### SQL read_files() Special Features

* **`_metadata` column**: Automatically available, contains:
  - `file_path`: Full path to source file
  - `file_name`: Name of the file
  - `file_size`: Size in bytes
  - `file_modification_time`: When file was last modified

### Performance Tips

✓ **Use `inferSchema=true` cautiously** - It requires scanning data twice  
✓ **Define explicit schema** for production workloads (faster than inferSchema)  
✓ **Use appropriate `mode`** - DROPMALFORMED for data quality, PERMISSIVE for debugging  
✓ **Partition large datasets** - Read only necessary files using path patterns  
✓ **Cache DataFrames** if reading same data multiple times

### Common Use Cases

**Standard CSV with header:**
```python
df = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
```

**Tab-delimited file:**
```python
df = spark.read.option("delimiter", "\t").option("header", "true").csv(path)
```

**Handle NULL values:**
```python
df = spark.read.option("nullValue", "NA").option("header", "true").csv(path)
```

**Strict parsing (fail on errors):**
```python
df = spark.read.option("mode", "FAILFAST").option("header", "true").csv(path)
```

In [0]:
# Method 1: PySpark DataFrame API - Basic Read
# This is the simplest way to read CSV files using PySpark

df_basic = spark.read.csv("/Volumes/dev_finance/staging/data_export/csv/")

# Display the first few rows
display(df_basic)

In [0]:
# Method 1: PySpark API - Reading with Header
# The 'header' option tells Spark to use the first row as column names

df_with_header = spark.read \
    .option("header", "true") \
    .csv("/Volumes/dev_finance/staging/data_export/csv/")

print("Schema with header:")
df_with_header.printSchema()
display(df_with_header)

In [0]:
# Method 1: PySpark API - Infer Schema Automatically
# The 'inferSchema' option automatically detects column data types
# Note: This requires an additional pass through the data

df_infer = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dev_finance/staging/data_export/csv/")

print("Schema with inferred types:")
df_infer.printSchema()
display(df_infer)

In [0]:
# Method 1: PySpark API - Complete Options
# Demonstrating multiple CSV reading options together

df_complete = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("quote", '"') \
    .option("escape", "\\") \
    .option("nullValue", "NULL") \
    .option("mode", "PERMISSIVE") \
    .csv("/Volumes/dev_finance/staging/data_export/csv/")

print("Complete DataFrame Info:")
print(f"Row count: {df_complete.count()}")
print(f"Column count: {len(df_complete.columns)}")
print(f"\nColumns: {df_complete.columns}")
display(df_complete)

In [0]:
%sql
-- Method 2: SQL read_files() Function - Basic Read
-- This is the SQL approach to read CSV files directly in queries

SELECT *
FROM read_files(
  '/Volumes/dev_finance/staging/data_export/csv/',
  format => 'csv'
)
LIMIT 100

In [0]:
%sql
-- Method 2: SQL read_files() - With Header
-- The 'header' option uses the first row as column names

SELECT *
FROM read_files(
  '/Volumes/dev_finance/staging/data_export/csv/',
  format => 'csv',
  header => true
)
LIMIT 100

In [0]:
%sql
-- Method 2: SQL read_files() - Infer Schema
-- Automatically detect column data types

SELECT *
FROM read_files(
  '/Volumes/dev_finance/staging/data_export/csv/',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 100

In [0]:
%sql
-- Method 2: SQL read_files() - Complete Options
-- Demonstrating multiple options together

SELECT *
FROM read_files(
  '/Volumes/dev_finance/staging/data_export/csv/',
  format => 'csv',
  header => true,
  inferSchema => true,
  delimiter => ',',
  quote => '"',
  escape => '\\',
  nullValue => 'NULL',
  mode => 'PERMISSIVE'
)
LIMIT 100

In [0]:
%sql
-- Method 2: SQL read_files() - Include File Metadata
-- The _metadata column provides information about the source file

SELECT 
  _metadata.file_path,
  _metadata.file_name,
  _metadata.file_size,
  _metadata.file_modification_time,
  *
FROM read_files(
  '/Volumes/dev_finance/staging/data_export/csv/',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 100